In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [5]:
system_message = "You are helpful assistant"

chat = ChatPromptTemplate.from_messages(
    [
        ("system",system_message),
        MessagesPlaceholder(variable_name="history"),
        ("user","{input}")
    ]
)

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

def get_session_history(session_id:str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [7]:
model = ChatOpenAI(
    model = "z-ai/glm-4.5-air:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=key,
    temperature=0
)

In [15]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain = chat| model

stateful_chat = RunnableWithMessageHistory(
    chain,
    get_session_history,
    history_messages_key="history",
    input_messages_key="input"
)


In [21]:
def response(user_ip,history):
    response = stateful_chat.invoke(
        {"input":user_ip},
        config = {"configurable": {"session_id": "default"}}
    )
    return response.content

In [22]:
import gradio as gr
gr.ChatInterface(fn = response).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
